# Silver Layer — CRM: Customer Info

Transforms the bronze `workspace.bronze.cust_info` table into `workspace.silver.customer_info` with meaningful, readable column names.

---

## Table of Contents — Transformations Applied

| # | Transformation | Description |
|---|---|---|
| 1 | **Column Renaming** | Renamed abbreviated columns to meaningful names (e.g., `cst_id` → `customer_id`, `cst_firstname` → `first_name`) |
| 2 | **Trim Whitespace** | Trimmed leading/trailing spaces from `first_name` and `last_name` |
| 3 | **Full Name Column** | Created new `full_name` column by concatenating trimmed `first_name` + space + trimmed `last_name` |
| 4 | **Marital Status Mapping** | Mapped codes to full words: `M` → `Married`, `S` → `Single` |
| 5 | **Gender Mapping** | Mapped codes to full words: `M` → `Male`, `F` → `Female` |

---

## Column Mapping (Bronze → Silver)

| Bronze Column | Silver Column | Transformation Applied |
|---|---|---|
| `cst_id` | `customer_id` | Renamed |
| `cst_key` | `customer_key` | Renamed |
| `cst_firstname` | `first_name` | Renamed + Trimmed |
| `cst_lastname` | `last_name` | Renamed + Trimmed |
| — | `full_name` | New column: `TRIM(first_name) + ' ' + TRIM(last_name)` |
| `cst_marital_status` | `marital_status` | Renamed + Mapped (`M`→`Married`, `S`→`Single`) |
| `cst_gndr` | `gender` | Renamed + Mapped (`M`→`Male`, `F`→`Female`) |
| `cst_create_date` | `create_date` | Renamed |

---

## Implementation

All transformations are implemented in **PySpark** (`pyspark.sql.functions`):
- `col()` — select and alias columns
- `trim()` — remove leading/trailing whitespace
- `concat()` + `lit()` — build full_name column
- `when()` + `otherwise()` — conditional mapping for marital_status and gender
- `df.write.mode("overwrite").saveAsTable()` — write to silver layer

In [0]:
from pyspark.sql.functions import col, trim, concat, lit, when

df = (spark.read.table("workspace.bronze.cust_info")
    .select(
        col("cst_id").alias("customer_id"),
        col("cst_key").alias("customer_key"),
        trim(col("cst_firstname")).alias("first_name"),
        trim(col("cst_lastname")).alias("last_name"),
        concat(trim(col("cst_firstname")), lit(" "), trim(col("cst_lastname"))).alias("full_name"),
        when(col("cst_marital_status") == "M", "Married")
            .when(col("cst_marital_status") == "S", "Single")
            .otherwise(col("cst_marital_status")).alias("marital_status"),
        when(col("cst_gndr") == "M", "Male")
            .when(col("cst_gndr") == "F", "Female")
            .otherwise(col("cst_gndr")).alias("gender"),
        col("cst_create_date").alias("create_date"),
    )
)

df.write.mode("overwrite").saveAsTable("workspace.silver.customer_info")

In [0]:
display(spark.read.table("workspace.silver.customer_info").limit(10))